# Lab 06 — Hough Transforms, SIFT, Wavelets, Boundaries (all 8 tasks)
**Lab-timing tasks:** 1 Screen detection (Hough lines) · 2 Asset tracking (SIFT) · 3 Sensor anomaly detection (wavelets)
**Lab tasks:** 4 Object recognition in video (SIFT) · 5 Panorama (SIFT + homography) · 6 Lane detection (Hough lines)
· 7 Coin detection & counting (Hough circles) · 8 Smart security system (boundaries in a zone, video)

One code cell per task. Run **Setup** first. Set the paths; missing files fall back to
synthetic images / videos / signals so every cell runs. Videos: `SHOW_LIVE = False`
shows sample frames inline (Jupyter) and writes the processed video to `output/`;
set it to `True` in a local Python run to get a live `cv2.imshow` window.

In [ ]:
# ===================== SETUP (run first) =====================
import os
import cv2
import numpy as np
import pandas as pd
import pywt
import matplotlib.pyplot as plt

PATHS = {
    "lab": "computer_lab.jpg",                                        # Task 1
    "assets_refs": {"monitor": "ref_monitor.jpg", "keyboard": "ref_keyboard.jpg", "cpu": "ref_cpu.jpg"},
    "assets_scene": "lab_scene.jpg",                                  # Task 2
    "sensor_csv": "sensor_data.csv",                                  # Task 3 (column 'value')
    "object_ref": "object.jpg", "object_video": "object_video.mp4",   # Task 4
    "pano": ["pano1.jpg", "pano2.jpg", "pano3.jpg"],                  # Task 5 (left -> right)
    "roads": ["road1.jpg", "road2.jpg"],                              # Task 6
    "coins": ["coins1.jpg", "coins2.jpg"],                            # Task 7
    "security_video": "security.mp4",                                 # Task 8
}
os.makedirs("output", exist_ok=True)
SHOW_LIVE = False


def show(imgs, titles, cols=3, cmap="gray"):
    rows = (len(imgs) + cols - 1) // cols
    plt.figure(figsize=(5.2 * cols, 4.2 * rows))
    for i, (im, t) in enumerate(zip(imgs, titles)):
        plt.subplot(rows, cols, i + 1)
        plt.imshow(im if im.ndim == 2 else cv2.cvtColor(im, cv2.COLOR_BGR2RGB), cmap=cmap)
        plt.title(t, fontsize=10); plt.axis("off")
    plt.tight_layout(); plt.show()


def pattern(h, w, seed, label=None, bg=None):
    """Distinctive textured patch (lots of corners/blobs -> many SIFT keypoints)."""
    rng = np.random.default_rng(seed)
    img = np.full((h, w, 3), bg if bg is not None else rng.integers(150, 255, 3), np.uint8)
    for _ in range(int(h * w / 900)):
        c = tuple(int(v) for v in rng.integers(0, 255, 3))
        x, y = int(rng.integers(0, w)), int(rng.integers(0, h))
        k = rng.integers(3)
        if k == 0: cv2.circle(img, (x, y), int(rng.integers(3, 14)), c, -1)
        elif k == 1: cv2.rectangle(img, (x, y), (x + int(rng.integers(5, 30)), y + int(rng.integers(5, 30))), c, -1)
        else: cv2.line(img, (x, y), (x + int(rng.integers(-40, 40)), y + int(rng.integers(-40, 40))), c, 2)
    if label:
        cv2.putText(img, label, (8, h - 12), cv2.FONT_HERSHEY_SIMPLEX, min(w, h) / 140, (0, 0, 0), 2)
    cv2.rectangle(img, (0, 0), (w - 1, h - 1), (0, 0, 0), 3)
    return img


def paste_warped(scene, obj, H):
    """Warp obj with homography H and paste it onto scene (in place)."""
    sh, sw = scene.shape[:2]
    warped = cv2.warpPerspective(obj, H, (sw, sh))
    m = cv2.warpPerspective(np.full(obj.shape[:2], 255, np.uint8), H, (sw, sh))
    scene[m > 0] = warped[m > 0]
    return scene


def similarity_H(s, deg, tx, ty, w, h):
    """Homography that scales by s, rotates by deg about the object's centre, moves its centre to (tx, ty)."""
    t = np.deg2rad(deg); c, si = np.cos(t), np.sin(t)
    return np.array([[s * c, -s * si, tx], [s * si, s * c, ty], [0, 0, 1]]) @ \
        np.array([[1, 0, -w / 2], [0, 1, -h / 2], [0, 0, 1]])


SIFT = cv2.SIFT_create()
BF = cv2.BFMatcher(cv2.NORM_L2)          # L2 for SIFT's float descriptors


def ratio_matches(d1, d2, ratio=0.75):
    if d1 is None or d2 is None or len(d2) < 2:
        return []
    return [m for m, n in (p for p in BF.knnMatch(d1, d2, k=2) if len(p) == 2) if m.distance < ratio * n.distance]


print("Setup done | OpenCV", cv2.__version__, "| PyWavelets", pywt.__version__)

In [ ]:
# ===================== TASK 1 — Computer screen detection in a lab (Hough lines) =====================
# Find screen boundaries with HoughLinesP, report each screen ON/OFF and flag missing screens.
def synthetic_lab():
    img = np.full((420, 760, 3), (175, 170, 165), np.uint8)
    cv2.rectangle(img, (0, 190), (759, 215), (40, 70, 110), -1)            # desk row 1
    cv2.rectangle(img, (0, 390), (759, 419), (40, 70, 110), -1)            # desk row 2
    slots, status = [], []
    rng = np.random.default_rng(6)
    for r, y in enumerate([60, 260]):
        for c, x in enumerate([40, 220, 400, 580]):
            slots.append((x + 70, y + 50))
            if (r, c) == (1, 2):                                            # missing screen
                status.append("missing"); continue
            cv2.rectangle(img, (x, y), (x + 140, y + 100), (25, 25, 25), -1)   # bezel
            on = rng.random() < 0.6
            scr = (225, 160, 70) if on else (55, 55, 55)
            cv2.rectangle(img, (x + 8, y + 8), (x + 132, y + 92), scr, -1)
            if on:
                cv2.rectangle(img, (x + 20, y + 20), (x + 80, y + 60), (250, 250, 250), -1)
            cv2.rectangle(img, (x + 62, y + 100), (x + 78, y + 130), (30, 30, 30), -1)  # stand
            status.append("on" if on else "off")
    return img, slots, status

img = cv2.imread(PATHS["lab"])
slots, truth = None, None
if img is None:
    print("Lab photo not found -> synthetic computer lab (2 rows x 4 desks)")
    img, slots, truth = synthetic_lab()
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 50, 150)

lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=30, minLineLength=30, maxLineGap=20)
line_mask = np.zeros_like(gray)
vis = img.copy()
if lines is not None:                                          # ALWAYS check None
    for x1, y1, x2, y2 in lines[:, 0]:
        ang = abs(np.degrees(np.arctan2(y2 - y1, x2 - x1))) % 180
        if ang < 10 or ang > 170 or 80 < ang < 100:          # keep horizontal & vertical (screen edges)
            cv2.line(line_mask, (x1, y1), (x2, y2), 255, 3)
            cv2.line(vis, (x1, y1), (x2, y2), (0, 0, 255), 2)
print("Hough segments:", 0 if lines is None else len(lines))

# Close gaps between Hough segments, then turn the line structure into rectangles
closed = cv2.morphologyEx(line_mask, cv2.MORPH_CLOSE, np.ones((15, 15), np.uint8))
cnts, _ = cv2.findContours(closed, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
boxes = [list(cv2.boundingRect(c)) for c in cnts]
boxes = [b for b in boxes if b[3] > 30 and b[2] * b[3] > 1500]   # drop thin desk lines / specks


def overlap(a, b, pad=5):
    return not (a[0] > b[0] + b[2] + pad or b[0] > a[0] + a[2] + pad or
                a[1] > b[1] + b[3] + pad or b[1] > a[1] + a[3] + pad)


merged = True                                                   # merge nested / split pieces
while merged:
    merged = False
    for i in range(len(boxes)):
        for j in range(i + 1, len(boxes)):
            if overlap(boxes[i], boxes[j]):
                a, b = boxes[i], boxes[j]
                x0, y0 = min(a[0], b[0]), min(a[1], b[1])
                x1, y1 = max(a[0] + a[2], b[0] + b[2]), max(a[1] + a[3], b[1] + b[3])
                boxes[i] = [x0, y0, x1 - x0, y1 - y0]; boxes.pop(j); merged = True; break
        if merged:
            break

hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
screens = []
for x, y, w, h in boxes:
    if w * h < 5000 or not (1.1 < w / h < 2.2):              # screen-like size & aspect ratio
        continue
    inner = hsv[y + h // 6:y + 5 * h // 6, x + w // 6:x + 5 * w // 6, 2]
    state = "ON" if inner.mean() > 120 else "OFF"            # brightness (V) of the display area
    screens.append((x, y, w, h, state))
    cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0) if state == "ON" else (255, 0, 0), 3)
    cv2.putText(vis, state, (x + 5, y - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

if slots is not None:                                          # expected desk positions
    for sx, sy in slots:
        if not any(x <= sx <= x + w and y <= sy <= y + h for x, y, w, h, _ in screens):
            cv2.putText(vis, "MISSING", (sx - 45, sy), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            print(f"ANOMALY: no screen at desk position {(sx, sy)}")
on = sum(s[4] == "ON" for s in screens)
print(f"Screens found: {len(screens)} | ON: {on} | OFF: {len(screens) - on}")
if truth: print("Ground truth:", truth)
show([img, edges, line_mask, vis], ["Computer lab", "Canny", "Horizontal/vertical Hough lines",
                                    "Screens: green=ON, blue=OFF"], cols=2)
cv2.imwrite("output/task1_screens.png", vis)

In [ ]:
# ===================== TASK 2 — Asset tracking in a computer lab (SIFT) =====================
# Recognise each known asset (one reference image per asset) inside a lab photo.
refs = {}
for i, (name, p) in enumerate(PATHS["assets_refs"].items()):
    im = cv2.imread(p)
    refs[name] = im if im is not None else pattern(160, 220, seed=100 + i, label=name.upper())
scene = cv2.imread(PATHS["assets_scene"])
if scene is None:
    print("Lab scene not found -> synthetic scene with monitor + cpu (keyboard absent)")
    scene = pattern(480, 720, seed=7, bg=(120, 120, 120))
    paste_warped(scene, refs["monitor"], similarity_H(1.1, 15, 200, 170, 220, 160))
    paste_warped(scene, refs["cpu"], similarity_H(0.7, -30, 540, 330, 220, 160))

g_scene = cv2.cvtColor(scene, cv2.COLOR_BGR2GRAY)
kp_s, d_s = SIFT.detectAndCompute(g_scene, None)
out, rows = scene.copy(), []
colors = [(0, 255, 0), (0, 0, 255), (255, 0, 255)]
for (name, ref), col in zip(refs.items(), colors):
    g_ref = cv2.cvtColor(ref, cv2.COLOR_BGR2GRAY)
    kp_r, d_r = SIFT.detectAndCompute(g_ref, None)
    good = ratio_matches(d_r, d_s)
    inliers, found = 0, False
    if len(good) >= 10:
        src = np.float32([kp_r[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
        dst = np.float32([kp_s[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
        H, mask = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
        if H is not None:
            inliers = int(mask.sum())
            found = inliers >= 10
            if found:
                h, w = g_ref.shape
                box = cv2.perspectiveTransform(np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2), H)
                cv2.polylines(out, [np.int32(box)], True, col, 4)
                x, y = np.int32(box[0, 0])
                cv2.putText(out, name, (x, max(20, y - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.9, col, 2)
    rows.append({"asset": name, "ref keypoints": len(kp_r), "good matches": len(good),
                 "RANSAC inliers": inliers, "status": "PRESENT" if found else "NOT FOUND"})
inventory = pd.DataFrame(rows)
print(inventory.to_string(index=False))
inventory.to_csv("output/task2_inventory.csv", index=False)
show(list(refs.values()) + [out], [f"Reference: {n}" for n in refs] + ["Identified assets"], cols=4)
cv2.imwrite("output/task2_assets.png", out)

In [ ]:
# ===================== TASK 3 — Anomaly detection in sensor data (wavelets) =====================
if os.path.exists(PATHS["sensor_csv"]):
    data = pd.read_csv(PATHS["sensor_csv"])["value"].to_numpy(float)
    true_idx = None
else:
    print("CSV not found -> synthetic sensor signal with injected anomalies")
    rng = np.random.default_rng(0)
    n = 1000
    data = np.sin(2 * np.pi * np.arange(n) / 100) + 0.3 * rng.normal(size=n)
    true_idx = np.array([200, 450, 720, 880])
    data[true_idx] += np.array([3.0, -2.8, 3.5, -3.2])
    pd.DataFrame({"value": data}).to_csv("output/task3_sensor_data.csv", index=False)

wavelet, level = "db4", 4
coeffs = pywt.wavedec(data, wavelet, level=level)            # [cA4, cD4, cD3, cD2, cD1]
sigma = np.median(np.abs(coeffs[-1])) / 0.6745               # noise estimate from finest details
thr = sigma * np.sqrt(2 * np.log(len(data)))                 # universal threshold
den_coeffs = [coeffs[0]] + [pywt.threshold(c, thr, mode="soft") for c in coeffs[1:]]
denoised = pywt.waverec(den_coeffs, wavelet)[:len(data)]     # trim: output can be 1 longer
residuals = data - denoised
k = 3
anomalies = np.where(np.abs(residuals) > k * np.std(residuals))[0]
print(f"threshold = {thr:.3f} | anomalies (|residual| > {k}·std): {anomalies.tolist()}")
if true_idx is not None:
    print("injected at:", true_idx.tolist())

x = np.arange(len(data))
plt.figure(figsize=(12, 7))
plt.subplot(2, 1, 1)
plt.plot(x, data, label="Sensor Data"); plt.plot(x, denoised, "--", label="Denoised Signal")
plt.legend(); plt.title("Sensor Data and Denoised Signal")
plt.subplot(2, 1, 2)
plt.plot(x, residuals, label="Residuals")
plt.scatter(anomalies, residuals[anomalies], color="red", zorder=3, label="Anomalies")
plt.legend(); plt.title("Residuals and Detected Anomalies")
plt.tight_layout(); plt.savefig("output/task3_anomalies.png"); plt.show()

In [ ]:
# ===================== TASK 4 — Object recognition in video (SIFT) =====================
# Locate a reference object in every frame despite scale, rotation and partial occlusion.
def synthetic_object_video(path, ref, n=60):
    rng = np.random.default_rng(9)
    bg = pattern(360, 480, seed=11, bg=(140, 140, 140))
    out = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), 15, (480, 360))
    h, w = ref.shape[:2]
    for i in range(n):
        f = bg.copy()
        s, ang = 0.6 + 0.4 * np.sin(i / 10) ** 2, 3 * i            # changing scale & rotation
        paste_warped(f, ref, similarity_H(s, ang, 140 + 3 * i, 180 + 40 * np.sin(i / 8), w, h))
        if 20 <= i < 40:                                            # partial occlusion
            cv2.rectangle(f, (150 + 3 * i, 120), (230 + 3 * i, 260), (60, 60, 200), -1)
        out.write(np.clip(f + rng.normal(0, 4, f.shape), 0, 255).astype(np.uint8))
    out.release()

ref = cv2.imread(PATHS["object_ref"])
if ref is None:
    ref = pattern(150, 200, seed=42, label="OBJECT")
if not os.path.exists(PATHS["object_video"]):
    print("Video not found -> writing a synthetic test video")
    synthetic_object_video(PATHS["object_video"], ref)

g_ref = cv2.cvtColor(ref, cv2.COLOR_BGR2GRAY)
kp_r, d_r = SIFT.detectAndCompute(g_ref, None)                     # reference features ONCE
h, w = g_ref.shape
corners = np.float32([[0, 0], [w, 0], [w, h], [0, h]]).reshape(-1, 1, 2)

cap = cv2.VideoCapture(PATHS["object_video"])                      # or a list of test images
fps = cap.get(cv2.CAP_PROP_FPS) or 15
W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter("output/task4_recognition.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
found_count, i, samples = 0, 0, []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    kp_f, d_f = SIFT.detectAndCompute(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), None)
    good = ratio_matches(d_r, d_f)
    label = f"matches {len(good)}"
    if len(good) >= 10:
        src = np.float32([kp_r[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
        dst = np.float32([kp_f[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
        Hm, mask = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
        if Hm is not None and mask.sum() >= 8:
            cv2.polylines(frame, [np.int32(cv2.perspectiveTransform(corners, Hm))], True, (0, 255, 0), 3)
            label += f", inliers {int(mask.sum())}: FOUND"; found_count += 1
    cv2.putText(frame, label, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    writer.write(frame)
    if SHOW_LIVE:
        cv2.imshow("Object recognition", frame)
        if cv2.waitKey(int(1000 / fps)) & 0xFF == ord("q"): break
    elif i in (5, 25, 45):
        samples.append(frame.copy())
    i += 1
cap.release(); writer.release()
if SHOW_LIVE: cv2.destroyAllWindows()
print(f"Object located in {found_count}/{i} frames -> output/task4_recognition.mp4")
show([ref] + samples, ["Reference object"] + [f"Frame sample {j + 1}" for j in range(len(samples))], cols=4)

In [ ]:
# ===================== TASK 5 — Panoramic image (SIFT stitching) =====================
def stitch(img1, img2, ratio=0.75):
    """Warp img2 (to the RIGHT of img1) into img1's plane and overlay."""
    g1, g2 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY), cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
    k1, d1 = SIFT.detectAndCompute(g1, None)
    k2, d2 = SIFT.detectAndCompute(g2, None)
    good = ratio_matches(d2, d1, ratio)                               # img2 -> img1
    if len(good) < 4:
        raise RuntimeError("Not enough matches to stitch")
    src = np.float32([k2[m.queryIdx].pt for m in good]).reshape(-1, 1, 2)
    dst = np.float32([k1[m.trainIdx].pt for m in good]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(src, dst, cv2.RANSAC, 5.0)
    print(f"  matches {len(good)}, inliers {int(mask.sum())}")
    pano = cv2.warpPerspective(img2, H, (img1.shape[1] + img2.shape[1], max(img1.shape[0], img2.shape[0])))
    region = pano[:img1.shape[0], :img1.shape[1]]
    region[img1.sum(axis=2) > 0] = img1[img1.sum(axis=2) > 0]         # keep img1 where it has content
    x, y, w, h = cv2.boundingRect(cv2.findNonZero(cv2.cvtColor(pano, cv2.COLOR_BGR2GRAY)))
    return pano[y:y + h, x:x + w]                                     # crop black border

imgs = [cv2.imread(p) for p in PATHS["pano"]]
if any(im is None for im in imgs):
    print("Pano photos not found -> 3 overlapping synthetic views of one wide scene")
    wide = pattern(380, 1100, seed=21)
    imgs = [wide[:, 0:450].copy(), wide[:, 320:770].copy(), wide[:, 640:1090].copy()]
    imgs[1] = cv2.warpAffine(imgs[1], cv2.getRotationMatrix2D((225, 190), 2, 1.0), (450, 380))  # camera wobble
pano = imgs[0]
for j, nxt in enumerate(imgs[1:], 2):
    print(f"Stitching image {j}:")
    pano = stitch(pano, nxt)
print("Panorama size:", pano.shape)
show(imgs, [f"Input {j + 1}" for j in range(len(imgs))], cols=len(imgs))
show([pano], ["Panorama"], cols=1)
cv2.imwrite("output/task5_panorama.png", pano)

In [ ]:
# ===================== TASK 6 — Lane detection (Hough lines) =====================
def synthetic_road(shift=0, seed=0):
    h, w = 360, 640
    img = np.zeros((h, w, 3), np.uint8); img[:h // 2] = (200, 170, 120); img[h // 2:] = (60, 110, 60)
    vx = w // 2 + shift
    cv2.fillPoly(img, [np.array([[0, h], [vx - 40, h // 2], [vx + 40, h // 2], [w, h]], np.int32)], (80, 80, 80))
    cv2.line(img, (90, h), (vx - 15, h // 2 + 10), (255, 255, 255), 8)
    for y in range(h, h // 2 + 10, -50):
        t0, t1 = (h - y) / (h / 2 - 10), (h - y + 28) / (h / 2 - 10)
        cv2.line(img, (int(560 + (vx + 15 - 560) * t0), y), (int(560 + (vx + 15 - 560) * t1), y - 28), (0, 220, 255), 8)
    return np.clip(img + np.random.default_rng(seed).normal(0, 6, img.shape), 0, 255).astype(np.uint8)


def detect_lanes(img):
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 50, 150)
    roi = np.zeros_like(edges)
    top = int(h * 0.55)
    cv2.fillPoly(roi, [np.array([[0, h], [int(w * 0.45), top], [int(w * 0.55), top], [w, h]], np.int32)], 255)
    masked = cv2.bitwise_and(edges, roi)                              # keep road region only
    segs = cv2.HoughLinesP(masked, 1, np.pi / 180, threshold=30, minLineLength=20, maxLineGap=60)
    out, sides = img.copy(), {"left": [], "right": []}
    if segs is not None:                                              # ALWAYS check None
        for x1, y1, x2, y2 in segs[:, 0]:
            slope = (y2 - y1) / (x2 - x1 + 1e-6)
            if abs(slope) < 0.4:                                      # drop near-horizontal lines
                continue
            sides["left" if slope < 0 else "right"] += [(x1, y1), (x2, y2)]
    for side, pts in sides.items():
        if len(pts) >= 2:
            pts = np.array(pts)
            m, b = np.polyfit(pts[:, 1], pts[:, 0], 1)                # x = m*y + b
            cv2.line(out, (int(m * h + b), h), (int(m * top + b), top), (0, 0, 255), 8)
    overlay = cv2.addWeighted(img, 0.4, out, 0.6, 0)
    return masked, overlay, {k: len(v) // 2 for k, v in sides.items()}


roads = [cv2.imread(p) for p in PATHS["roads"]]
if any(r is None for r in roads):
    print("Road images not found -> 2 synthetic road frames")
    roads = [synthetic_road(0, 1), synthetic_road(60, 2)]
panels, titles = [], []
for j, r in enumerate(roads, 1):
    masked, overlay, counts = detect_lanes(r)
    print(f"Image {j}: segments per side {counts}")
    panels += [r, masked, overlay]; titles += [f"Road {j}", "Canny in ROI", "Detected lanes"]
    cv2.imwrite(f"output/task6_lanes_{j}.png", overlay)
show(panels, titles, cols=3)

In [ ]:
# ===================== TASK 7 — Coin detection and counting (Hough circles) =====================
def synthetic_coins(seed):
    rng = np.random.default_rng(seed)
    img = np.full((400, 560, 3), (60, 90, 60), np.uint8)
    placed = []
    while len(placed) < int(rng.integers(6, 10)):
        r = int(rng.integers(22, 50)); x, y = int(rng.integers(r, 560 - r)), int(rng.integers(r, 400 - r))
        if all(np.hypot(x - a, y - b) > r + c + 4 for a, b, c in placed):
            placed.append((x, y, r))
            cv2.circle(img, (x, y), r, tuple(int(v) for v in rng.integers(130, 230, 3)), -1)
            cv2.circle(img, (x, y), r - 5, (90, 90, 90), 2)
    return np.clip(img + rng.normal(0, 6, img.shape), 0, 255).astype(np.uint8), len(placed)

images = [cv2.imread(p) for p in PATHS["coins"]]
truths = [None] * len(images)
if any(im is None for im in images):
    print("Coin images not found -> 2 synthetic coin images")
    (a, ta), (b, tb) = synthetic_coins(3), synthetic_coins(4)
    images, truths = [a, b], [ta, tb]
outs, titles = [], []
for j, (img, t) in enumerate(zip(images, truths), 1):
    g = cv2.medianBlur(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), 5)     # blurred gray, NOT Canny output
    circles = cv2.HoughCircles(g, cv2.HOUGH_GRADIENT, dp=1.2, minDist=40,
                               param1=100, param2=30, minRadius=15, maxRadius=70)
    out = img.copy(); n = 0
    if circles is not None:
        circles = np.uint16(np.around(circles))
        n = circles.shape[1]
        for x, y, r in circles[0]:
            cv2.circle(out, (x, y), r, (0, 255, 0), 2)              # outline
            cv2.circle(out, (x, y), 2, (0, 0, 255), 3)              # centre
    cv2.putText(out, f"Coins: {n}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
    print(f"Image {j}: {n} coins detected" + (f" (true {t})" if t else ""))
    outs += [img, out]; titles += [f"Image {j}", f"Detected: {n}"]
    cv2.imwrite(f"output/task7_coins_{j}.png", out)
show(outs, titles, cols=2)
print("Tuning: missing coins -> lower param2 or widen min/maxRadius; extra circles -> raise param2 or minDist.")

In [ ]:
# ===================== TASK 8 — Smart security system (boundaries in a security zone) =====================
# Background subtraction -> contours (object boundaries) -> alarm if an object is inside the zone.
def synthetic_security_video(path, n=120):
    rng = np.random.default_rng(12)
    room = np.full((360, 480, 3), (150, 145, 140), np.uint8)
    cv2.rectangle(room, (0, 260), (479, 359), (90, 100, 110), -1)          # floor
    cv2.rectangle(room, (380, 80), (450, 260), (60, 60, 90), -1)            # cabinet
    out = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), 20, (480, 360))
    for i in range(n):
        f = room.copy()
        if i >= 50:                                                          # object walks in from the left
            x = min(-60 + 6 * (i - 50), 220)
            cv2.rectangle(f, (x, 190), (x + 60, 300), (30, 60, 200), -1)
        out.write(np.clip(f + rng.normal(0, 3, f.shape), 0, 255).astype(np.uint8))
    out.release()

if not os.path.exists(PATHS["security_video"]):
    print("Security video not found -> writing a synthetic CCTV clip")
    synthetic_security_video(PATHS["security_video"])

zone = np.array([[180, 150], [360, 150], [360, 330], [180, 330]], np.int32)   # sensitive area
bgs = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50, detectShadows=True)
cap = cv2.VideoCapture(PATHS["security_video"])
fps = cap.get(cv2.CAP_PROP_FPS) or 20
W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter("output/task8_security.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
alarm_frames, i, samples = [], 0, []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    fg = bgs.apply(frame)
    fg = cv2.threshold(fg, 200, 255, cv2.THRESH_BINARY)[1]           # drop shadows (value 127)
    fg = cv2.morphologyEx(fg, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))
    cnts, _ = cv2.findContours(fg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.polylines(frame, [zone], True, (255, 0, 0), 2)
    alarm = False
    for c in cnts:
        if cv2.contourArea(c) < 500:                                  # ignore noise blobs
            continue
        x, y, w, h = cv2.boundingRect(c)
        cx, cy = x + w // 2, y + h // 2
        inside = cv2.pointPolygonTest(zone, (float(cx), float(cy)), False) >= 0
        cv2.drawContours(frame, [c], -1, (0, 0, 255) if inside else (0, 255, 0), 2)   # object boundary
        alarm |= inside
    if alarm:
        alarm_frames.append(i)
        cv2.putText(frame, "ALARM: object in security zone", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
    writer.write(frame)
    if SHOW_LIVE:
        cv2.imshow("Security", frame)
        if cv2.waitKey(int(1000 / fps)) & 0xFF == ord("q"): break
    elif i in (30, 62, 80, 110):
        samples.append(frame.copy())
    i += 1
cap.release(); writer.release()
if SHOW_LIVE: cv2.destroyAllWindows()
print(f"Frames: {i} | alarm in {len(alarm_frames)} frames"
      + (f", first alarm at frame {alarm_frames[0]}" if alarm_frames else ""))
show(samples, [f"Frame {k}" for k in (30, 62, 80, 110)][:len(samples)], cols=2)